# NewsQA RAG - Phase 2 End-to-End Baseline (Kaggle)
One locked RAG baseline on resolved development questions. Retrieval uses the Round 3 chunk winner with BGE-M3 sparse retrieval and the MiniLM cross-encoder; generation uses Gemini 3.1 Flash-Lite and RAGAS judging uses Gemini 3.7 Flash.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time, zipfile
REPO_URL='https://github.com/ThomasdeCarpio/Text-Mining---NewsQA-RAG.git'
REPO_COMMIT='ca60e3fa73d5a2d241d9aa06c977b7c98abce168'
HF_REPO_ID='ThomasAnderson2009/newsqa-rag-evaluation'
HF_REVISION='v1.0.0'
RUN_MODE='smoke'  # Keep 'smoke' for the 5-question validation; change to 'full' only after review.
assert RUN_MODE in {'smoke','full'}
ROUND3_CSV=None  # Optional explicit path; otherwise the notebook searches Kaggle inputs.
LOCKED_CHUNK_SIZE=512
LOCKED_CHUNK_OVERLAP=64
GENERATOR_MODEL='gemini-3.1-flash-lite'
JUDGE_MODEL='gemini-3.7-flash'
RERANKER_MODEL='cross-encoder/ms-marco-MiniLM-L-6-v2'
DEVELOPMENT_ARTICLES=50
TOP_K=20
RERANK_TOP_N=5
GENERATOR_MAX_TOKENS=512
GENERATOR_MIN_INTERVAL_SECONDS=4.2  # Conservative pacing for a 15 RPM free project.
JUDGE_PILOT_QUESTIONS=5 if RUN_MODE=='smoke' else 25
JUDGE_BATCH_SIZE=5
JUDGE_MAX_WORKERS=1
MAX_ATTEMPTS=5
EVAL_QUESTIONS=5 if RUN_MODE=='smoke' else None
EXPERIMENT_ID='phase2-e2e-baseline-smoke' if RUN_MODE=='smoke' else 'phase2-e2e-baseline'
CHECKPOINT_NAME=f'phase2_{RUN_MODE}_resume_checkpoint.zip'
AUTO_RESTORE_FROM_INPUT=True
KAGGLE_WORKING=Path('/kaggle/working')
KAGGLE_INPUT=Path('/kaggle/input')
PROJECT_ROOT=KAGGLE_WORKING/'Text-Mining---NewsQA-RAG'
WORK_ROOT=KAGGLE_WORKING/'newsqa_phase2'
DATA_ROOT=WORK_ROOT/'data'
INDEX_ROOT=WORK_ROOT/'index'
EXPERIMENTS=WORK_ROOT/'experiments'
SPECS=WORK_ROOT/'specs'
CACHE=WORK_ROOT/'retrieval_cache'
RESULTS=WORK_ROOT/'results'/RUN_MODE
LOGS=WORK_ROOT/'logs'/RUN_MODE


## 1. Kaggle setup
Enable Internet and one GPU. The notebook defaults to `RUN_MODE='smoke'`, which performs the complete pipeline on exactly five questions and exports reviewable results. Add and enable three private Kaggle secrets: `HF_TOKEN`, `GEMINI_API_KEY_1` (free generator project), and `GEMINI_API_KEY` (paid judge project). The keys should belong to separate Google projects if separate quotas are required. Attach the Phase 1 Round 3 output containing `round3.csv` as a Kaggle input. The held-out final-test partition is not used.

In [ ]:
from kaggle_secrets import UserSecretsClient
secrets=UserSecretsClient()
os.environ['HF_TOKEN']=secrets.get_secret('HF_TOKEN') or ''
GENERATOR_API_KEY=secrets.get_secret('GEMINI_API_KEY_1') or ''
JUDGE_API_KEY=secrets.get_secret('GEMINI_API_KEY') or ''
assert os.environ['HF_TOKEN'], 'Add and enable the Kaggle secret HF_TOKEN'
assert GENERATOR_API_KEY, 'Add and enable GEMINI_API_KEY_1 for free-tier generation'
assert JUDGE_API_KEY, 'Add and enable GEMINI_API_KEY for paid RAGAS judging'
assert GENERATOR_API_KEY != JUDGE_API_KEY, 'Generator and judge keys must be different'
assert not REPO_COMMIT.startswith('SET_TO_'), 'Pin REPO_COMMIT after committing the Phase 2 implementation'
os.environ['HF_HOME']=str(KAGGLE_WORKING/'hf_cache')
os.environ.update({'TOKENIZERS_PARALLELISM':'false','OMP_NUM_THREADS':'1','MKL_NUM_THREADS':'1','PYTHONUNBUFFERED':'1','CUDA_VISIBLE_DEVICES':'0','LANGCHAIN_TRACING_V2':'false','LANGSMITH_TRACING':'false'})
if AUTO_RESTORE_FROM_INPUT and not WORK_ROOT.exists():
    checkpoints=sorted(KAGGLE_INPUT.rglob(CHECKPOINT_NAME),key=lambda path:path.stat().st_mtime,reverse=True)
    if checkpoints:
        WORK_ROOT.mkdir(parents=True,exist_ok=True); shutil.unpack_archive(checkpoints[0],WORK_ROOT); print('Restored:',checkpoints[0],flush=True)
for path in [DATA_ROOT,INDEX_ROOT,EXPERIMENTS,SPECS,CACHE,RESULTS,LOGS]: path.mkdir(parents=True,exist_ok=True)
if not PROJECT_ROOT.exists(): subprocess.run(['git','clone','--filter=blob:none',REPO_URL,str(PROJECT_ROOT)],check=True)
subprocess.run(['git','fetch','--depth=1','origin',REPO_COMMIT],cwd=PROJECT_ROOT,check=True,timeout=180)
subprocess.run(['git','checkout','--detach',REPO_COMMIT],cwd=PROJECT_ROOT,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt'],cwd=PROJECT_ROOT,check=True)
import pandas as pd, torch, yaml
from IPython.display import display
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator'
print('GPU:',torch.cuda.get_device_name(0),round(torch.cuda.get_device_properties(0).total_memory/2**30,1),'GiB')
print('Pinned commit:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=PROJECT_ROOT,text=True).strip())
print('Run mode:',RUN_MODE,'| questions:',EVAL_QUESTIONS or 281)


In [ ]:
def write_checkpoint():
    target=KAGGLE_WORKING/CHECKPOINT_NAME
    temporary=target.with_suffix('.zip.tmp')
    with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_DEFLATED) as archive:
        for name in ['experiments','results','specs','retrieval_cache','logs']:
            root=WORK_ROOT/name
            if root.exists():
                for path in root.rglob('*'):
                    if path.is_file(): archive.write(path,path.relative_to(WORK_ROOT))
    temporary.replace(target); print('Resume checkpoint:',target,round(target.stat().st_size/2**20,1),'MiB',flush=True); return target
def run_command(command,label,env_overrides=None):
    log_path=LOGS/f'{label}_{time.strftime("%Y%m%d_%H%M%S")}.log'
    command=[str(value) for value in command]
    print('$',' '.join(command),flush=True); print('Log:',log_path,flush=True)
    with log_path.open('w',encoding='utf-8') as log:
        child_env=os.environ.copy(); child_env.update(env_overrides or {})
        process=subprocess.Popen(command,cwd=PROJECT_ROOT,env=child_env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace',bufsize=1)
        for line in process.stdout:
            print(line,end='',flush=True); log.write(line); log.flush()
        returncode=process.wait()
    if returncode:
        write_checkpoint(); raise subprocess.CalledProcessError(returncode,command)
    return log_path
def locate_round3_csv():
    if ROUND3_CSV:
        path=Path(ROUND3_CSV); assert path.exists(), path; return path
    candidates=sorted(KAGGLE_INPUT.rglob('round3.csv'),key=lambda path:path.stat().st_mtime,reverse=True)
    if not candidates: raise FileNotFoundError('Attach the Round 3 output containing round3.csv, or set ROUND3_CSV')
    return candidates[0]
def disk_status():
    usage=shutil.disk_usage(KAGGLE_WORKING)
    value={'free_gib':round(usage.free/2**30,2),'used_gib':round(usage.used/2**30,2),'total_gib':round(usage.total/2**30,2)}
    print('Disk:',value,flush=True); return value


## 2. Lock the Phase 1 retrieval configuration
The retrieval pipeline is locked from the completed Phase 1 protocol: resolved questions, BGE-M3 sparse retrieval, MiniLM reranking, and recursive chunking 512/64. The cell validates that the corresponding Round 3 row exists and records the disclosed practical-equivalence amendment.

In [ ]:
round3_path=locate_round3_csv(); round3=pd.read_csv(round3_path)
required={'variant','reranker_model','index','retrieval.mrr@5.mean','retrieval.ndcg@5.mean','retrieval.hit_rate@5.mean','latency.total.p50_ms'}
missing=required-set(round3.columns); assert not missing, f'Round 3 CSV missing columns: {sorted(missing)}'
eligible=round3[(round3['variant']=='resolved') & (round3['reranker_model']==RERANKER_MODEL)].copy()
assert len(eligible)==3, f'Expected three resolved MiniLM Round 3 rows, found {len(eligible)}'
eligible=eligible.sort_values('index')
display(eligible[['index','retrieval.hit_rate@5.mean','retrieval.mrr@5.mean','retrieval.ndcg@5.mean','latency.total.p50_ms']])
selected=eligible[eligible['index']==f'chunk_{LOCKED_CHUNK_SIZE}_{LOCKED_CHUNK_OVERLAP}']
assert len(selected)==1, 'Locked 512/64 Round 3 row is missing or duplicated'
winner=selected.iloc[0].to_dict(); CHUNK_SIZE,CHUNK_OVERLAP=LOCKED_CHUNK_SIZE,LOCKED_CHUNK_OVERLAP
selection={'source_csv':str(round3_path),'selected_index':winner['index'],'chunk_size':CHUNK_SIZE,'chunk_overlap':CHUNK_OVERLAP,'reranker_model':RERANKER_MODEL,'retriever':'sparse','sparse_model':'BAAI/bge-m3','top_k':TOP_K,'rerank_top_n':RERANK_TOP_N,'selection_metrics':{key:winner[key] for key in ['retrieval.hit_rate@5.mean','retrieval.mrr@5.mean','retrieval.ndcg@5.mean','latency.total.p50_ms']},'protocol_amendment':{'status':'declared_before_phase2_generation','practical_mrr_tie_threshold':0.001,'reason':'512/64 and 1024/128 MiniLM MRR differ by 0.00024 with overlapping 95% CIs; 512/64 has better Hit@5, Recall@5, NDCG@5, and P50 latency'}}
(RESULTS/'retrieval_lock.json').write_text(json.dumps(selection,indent=2,sort_keys=True)+'\n',encoding='utf-8')
print(json.dumps(selection,indent=2))


## 3. API and model preflight
This makes one small request to each locked model using its assigned key before the expensive corpus build. Gemini 3.7 uses its default medium thinking and receives no deprecated sampling parameters.

In [ ]:
from openai import OpenAI
def verify_assigned_model(api_key,model):
    client=OpenAI(api_key=api_key,base_url='https://generativelanguage.googleapis.com/v1beta/openai/')
    request={'model':model,'messages':[{'role':'user','content':'Reply with exactly OK'}],'max_tokens':10}
    if not model.startswith('gemini-3.7'): request['temperature']=0.0
    response=client.chat.completions.create(**request)
    assert response.choices and response.choices[0].message.content, f'Empty preflight response from {model}'
    print('Model access verified:',model)
verify_assigned_model(GENERATOR_API_KEY,GENERATOR_MODEL)
verify_assigned_model(JUDGE_API_KEY,JUDGE_MODEL)
disk_status()


## 4. Materialize and index the locked corpus
The raw private Hugging Face release is rebuilt with the selected chunking. Only one BGE-M3 sparse index is created; the 11,064-article corpus and reviewed resolved questions remain fixed. Existing complete artifacts are reused.

In [ ]:
base_config=yaml.safe_load((PROJECT_ROOT/'configs/config.yaml').read_text())
base_config.setdefault('llm',{}).update({'model':GENERATOR_MODEL,'temperature':0.0,'max_tokens':GENERATOR_MAX_TOKENS})
phase2_base=SPECS/'phase2_base_config.yaml'; phase2_base.write_text(yaml.safe_dump(base_config,sort_keys=False),encoding='utf-8')
variant_name=f'chunk_{CHUNK_SIZE}_{CHUNK_OVERLAP}_recursive'; variant_root=DATA_ROOT/variant_name
chunks_path=variant_root/'final_deduplicated/chunks.jsonl'
if not chunks_path.exists():
    run_command([sys.executable,'-u','scripts/build_ablation_datasets.py','--repo-id',HF_REPO_ID,'--revision',HF_REVISION,'--base-config',phase2_base,'--output-base',DATA_ROOT,'--chunk-sizes',CHUNK_SIZE,'--chunk-overlaps',CHUNK_OVERLAP,'--strategies','recursive','--db-path-base',DATA_ROOT/'temporary_db','--skip-vector-index'],'materialize')
index_manifest=INDEX_ROOT/'index_manifest.json'
if not index_manifest.exists():
    run_command([sys.executable,'-u','scripts/build_retrieval_models_index.py','--chunks-path',chunks_path,'--base-config',phase2_base,'--base-variant-manifest',variant_root/'manifests/deduplicated.variant.json','--output-dir',INDEX_ROOT,'--sparse-ids','bge_m3_sparse','--skip-dense','--device','cuda'],'build_bge_m3')
index_item=json.loads(index_manifest.read_text())['sparse_indexes']['bge_m3_sparse']
testset=variant_root/'final_deduplicated/testset_resolved.jsonl'
profile='bge_m3_minilm'
spec={'schema_version':1,'experiment':{'id':EXPERIMENT_ID,'name':f'Phase 2 end-to-end baseline ({RUN_MODE})'},'output_dir':str(EXPERIMENTS),'seed':42,'dataset':{'article_field':'article_key','development_articles':DEVELOPMENT_ARTICLES,'indexes':{profile:{'config':index_item['config_path'],'variant_manifest':index_item['variant_manifest'],'testsets':{'resolved':str(testset)}}}},'fixed':{'index':profile,'variant':'resolved','partition':'development','retrieval_only':False,'retriever':'sparse','reranker':'cross-encoder','reranker_model':RERANKER_MODEL,'generator_model':GENERATOR_MODEL,'top_k':TOP_K,'rerank_top_n':RERANK_TOP_N},'runtime':{'max_attempts':MAX_ATTEMPTS,'retry_failed':True,'progress':True,'warmup_queries':1,'generation_min_interval_seconds':GENERATOR_MIN_INTERVAL_SECONDS,'shared_retrieval_cache':str(CACHE),**({'n_eval':EVAL_QUESTIONS} if EVAL_QUESTIONS else {})},'judge':{'enabled':False},'pricing':{'provider':'gemini','currency':'USD','input_per_million':0.25,'output_per_million':1.50,'judge':{'model':JUDGE_MODEL,'input_per_million':0.75,'output_per_million':3.75,'price_window':'through_2026-12-31'}},'summary':{'metrics':['retrieval.hit_rate@5','retrieval.mrr@5','retrieval.ndcg@5','qa.exact_match','qa.f1','citations.citation_validity','citations.citation_precision','citations.citation_recall','citations.citation_f1','ragas.faithfulness','ragas.answer_relevancy','ragas.context_precision','ragas.context_recall','ragas.answer_correctness'],'paired_metric':'ragas.answer_correctness','quality_metric':'ragas.answer_correctness.mean','latency_metric':'latency.total.p95_ms'}}
spec_path=SPECS/f'{EXPERIMENT_ID}.yaml'; spec_path.write_text(yaml.safe_dump(spec,sort_keys=False),encoding='utf-8')
sys.path.insert(0,str(PROJECT_ROOT/'backend'))
from newsqa_rag.experiments import build_article_partitions
partition=build_article_partitions({'resolved':testset},DEVELOPMENT_ARTICLES,42)
development_count=len(partition['partitions']['development']['question_ids']['resolved'])
assert development_count==281, f'Expected 281 development questions, got {development_count}'
assert EVAL_QUESTIONS is None or EVAL_QUESTIONS==5
print('Resolved development pool:',development_count,'| this run:',EVAL_QUESTIONS or development_count); print('Spec:',spec_path); disk_status()


## 5. Collect cited RAG answers
This runs retrieval, reranking, and generation for exactly five seeded questions in smoke mode, or all 281 resolved development questions in full mode. JSONL traces are append-only: rerunning the cell skips successful questions and resumes failed work.

In [ ]:
run_command([sys.executable,'-u','scripts/run_experiment.py',spec_path],'generation',{'GEMINI_API_KEY':GENERATOR_API_KEY})
experiment_dir=EXPERIMENTS/EXPERIMENT_ID
run_dirs=[path.parent for path in experiment_dir.glob('*/experiment_run.json')]
assert len(run_dirs)==1, f'Expected one baseline run, found {len(run_dirs)}'
RUN_DIR=run_dirs[0]
report=json.loads((RUN_DIR/'report.json').read_text()); display(report['coverage']); display(report.get('qa')); display(report.get('citations'))
expected_questions=EVAL_QUESTIONS or 281
assert report['coverage']['expected']==expected_questions, f"Expected {expected_questions} questions, got {report['coverage']['expected']}"
if report['coverage']['success_rate']<0.95: print('WARNING: generation success is below the 95% acceptance target; inspect attempts.jsonl before judging.')
print('Resumable run:',RUN_DIR); write_checkpoint()


## 6. RAGAS smoke/pilot
Smoke mode judges all five generated answers with all five RAGAS metrics. Full mode uses a seeded 25-question pilot. Both write into the same judge cache used by their respective run.

In [ ]:
judge_base=[sys.executable,'-u','scripts/judge_benchmark_predictions.py','--run-dir',RUN_DIR,'--judge-provider','gemini','--judge-model',JUDGE_MODEL,'--batch-size',JUDGE_BATCH_SIZE,'--max-workers',JUDGE_MAX_WORKERS,'--max-attempts',MAX_ATTEMPTS,'--seed',42,'--progress']
run_command(judge_base+['--n-eval',JUDGE_PILOT_QUESTIONS],'ragas_pilot',{'GEMINI_API_KEY':JUDGE_API_KEY})
run_command([sys.executable,'-u','scripts/score_benchmark_predictions.py','--run-dir',RUN_DIR],'score_pilot')
pilot_report=json.loads((RUN_DIR/'report.json').read_text()); display(pilot_report.get('ragas'))
assert pilot_report.get('ragas',{}).get('n_samples',0)==min(JUDGE_PILOT_QUESTIONS,pilot_report['coverage']['successful'])
if RUN_MODE=='smoke':
    run_command([sys.executable,'-u','scripts/summarize_experiments.py',experiment_dir],'summarize_smoke')
final_report=pilot_report
write_checkpoint()


## 7. Complete RAGAS judging (full mode only)
In smoke mode this cell does not make additional API requests. In full mode, run it after inspecting the 25-question pilot; it reuses completed judgments and evaluates only the remaining successful generations.

In [ ]:
if RUN_MODE=='full':
    run_command(judge_base+['--retry-failed'],'ragas_full',{'GEMINI_API_KEY':JUDGE_API_KEY})
    run_command([sys.executable,'-u','scripts/score_benchmark_predictions.py','--run-dir',RUN_DIR],'score_final')
    run_command([sys.executable,'-u','scripts/summarize_experiments.py',experiment_dir],'summarize')
    final_report=json.loads((RUN_DIR/'report.json').read_text())
else:
    print('Smoke mode: every successful answer from the five-question run was already judged; no additional API calls.')
display(final_report['coverage']); display(final_report.get('ragas'))
ragas_coverage=final_report.get('ragas',{}).get('coverage',0)
if ragas_coverage<0.95: print('WARNING: RAGAS coverage is below 95%; rerun this cell after resolving quota errors.')
write_checkpoint()


## 8. Baseline summary and exports
The table combines deterministic QA/citation metrics, RAGAS means and bootstrap confidence intervals, latency, coverage, token use, and estimated generation cost. Smoke mode also exports the raw five-question traces required for review.

In [ ]:
comparison=pd.read_csv(experiment_dir/'comparison.csv'); display(comparison.T)
score_rows=pd.read_json(RUN_DIR/'deterministic_scores.jsonl',lines=True); scored=pd.json_normalize(score_rows.to_dict('records'))
diagnostics={'pipeline_failures':int((scored['status']!='success').sum()),'retrieval_misses_at_5':int((scored['retrieval.hit_rate@5']==0).sum()),'answers_without_valid_citation':int((scored['citations.citation_validity']==0).sum()),'ragas_rows':int(scored.get('ragas.answer_correctness',pd.Series(dtype=float)).notna().sum())}
display(pd.DataFrame([diagnostics]))
if 'ragas.answer_correctness' in scored: display(scored.nsmallest(10,'ragas.answer_correctness')[['question_id','article_key','qa.f1','citations.citation_f1','ragas.faithfulness','ragas.answer_correctness']])
attempt_path=RUN_DIR/'attempts.jsonl'
if attempt_path.exists():
    attempts=pd.read_json(attempt_path,lines=True); failed_attempts=attempts[attempts['status']=='failed']; display(failed_attempts['stage'].value_counts().rename('failed_attempts'))
metric_rows=[]
for group in ['qa','citations','ragas']:
    for name,value in final_report.get(group,{}).items():
        if isinstance(value,(int,float)) and name not in {'n_samples','coverage'}: metric_rows.append({'group':group,'metric':name,'value':value})
metrics_frame=pd.DataFrame(metric_rows); display(metrics_frame)
import matplotlib.pyplot as plt, seaborn as sns
plt.figure(figsize=(10,5)); sns.barplot(data=metrics_frame,x='metric',y='value',hue='group'); plt.ylim(0,1); plt.xticks(rotation=35,ha='right'); plt.tight_layout(); plt.savefig(RESULTS/'phase2_quality_metrics.png',dpi=180); plt.show(); plt.close()
for name in ['run_manifest.json','environment.json','experiment_run.json','retrievals.jsonl','predictions.jsonl','attempts.jsonl','judge_results.jsonl','deterministic_scores.jsonl','report.json','report_summary.txt']:
    source=RUN_DIR/name
    if source.exists(): shutil.copy2(source,RESULTS/name)
shutil.copy2(experiment_dir/'comparison.csv',RESULTS/'comparison.csv'); shutil.copy2(spec_path,RESULTS/'experiment_spec.yaml')
smoke_manifest={'schema_version':1,'run_mode':RUN_MODE,'expected_questions':expected_questions,'generator_model':GENERATOR_MODEL,'generator_key_secret':'GEMINI_API_KEY_1','judge_model':JUDGE_MODEL,'judge_key_secret':'GEMINI_API_KEY','run_dir':str(RUN_DIR),'generated_at':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime())}
(RESULTS/f'{RUN_MODE}_manifest.json').write_text(json.dumps(smoke_manifest,indent=2,sort_keys=True)+'\n',encoding='utf-8')
archive_name=f'phase2_e2e_baseline_{RUN_MODE}_results'
archive=shutil.make_archive(str(KAGGLE_WORKING/archive_name),'zip',root_dir=RESULTS)
print('Compact results:',archive)
print('Full resumable artifacts:',experiment_dir)
print('Generation usage:',final_report.get('usage')); print('Estimated generation cost:',comparison.get('estimated_generation_cost_usd',pd.Series([None])).iloc[0]); disk_status()
